# Notebook 4: Audit de Fairness

## Credit Scoring avec IA Explicable (XAI)

### Objectifs
- Auditer la parité démographique (Demographic Parity)
- Auditer l'égalité des chances (Equalized Odds)
- Analyser les disparités par genre et âge
- Appliquer des techniques d'atténuation des biais
- Générer un rapport d'audit

In [ ]:
# Import des bibliothèques
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import des modules du projet
from src.data_loader import load_data
from src.preprocessing import DataPreprocessor, split_data
from src.models.xgboost_model import XGBoostModel
from src.fairness.fairness_audit import FairnessAuditor
from src.config import MODELS_DIR

print("✅ Import des bibliothèques réussi")

## 1. Chargement du Modèle et des Données

In [ ]:
# Charger les données
X, y = load_data()
preprocessor = DataPreprocessor()
X_transformed = preprocessor.fit_transform(X)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_transformed, y)

print(f"✅ Données chargées: {X_test.shape}")

In [ ]:
# Charger le modèle XGBoost
model = XGBoostModel()
model_path = MODELS_DIR / "xgboost_model.pkl"

if model_path.exists():
    model.load(str(model_path))
    print(f"✅ Modèle chargé depuis {model_path}")
else:
    model.train(X_train, y_train, X_val, y_val)
    model.save(str(model_path))
    print(f"✅ Modèle entraîné et sauvegardé")

## 2. Initialisation de l'Auditeur de Fairness

In [ ]:
# Créer l'auditeur de fairness
auditor = FairnessAuditor(model)

print("✅ Auditeur de fairness initialisé")

## 3. Audit de Parité Démographique par Genre

In [ ]:
# Audit de parité démographique par genre
gender_dp = auditor.audit_demographic_parity(
    X_test,
    y_test,
    X_test['gender']
)

print("\n📊 Audit de Parité Démographique par Genre:")
print(f"   - Différence: {gender_dp['difference']:.4f}")
print(f"   - Ratio: {gender_dp['ratio']:.4f}")
print(f"   - Interprétation: {gender_dp['interpretation']}")

print("\n📊 Taux de sélection par genre:")
for gender, rate in gender_dp['selection_rates'].items():
    print(f"   - {gender}: {rate:.4f}")

In [ ]:
# Visualiser les taux de sélection par genre
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
gender_df = pd.DataFrame({
    'Genre': list(gender_dp['selection_rates'].keys()),
    'Taux de sélection': list(gender_dp['selection_rates'].values())
})
sns.barplot(data=gender_df, x='Genre', y='Taux de sélection', ax=axes[0])
axes[0].set_title('Taux de Sélection par Genre')
axes[0].set_ylabel('Taux de sélection')
axes[0].set_ylim(0, 1)

# Pie chart
axes[1].pie(gender_dp['selection_rates'].values(), labels=gender_dp['selection_rates'].keys(),
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Répartition par Genre')

plt.tight_layout()
plt.show()

## 4. Audit d'Égalité des Chances par Genre

In [ ]:
# Audit d'égalité des chances par genre
gender_eo = auditor.audit_equalized_odds(
    X_test,
    y_test,
    X_test['gender']
)

print("\n📊 Audit d'Égalité des Chances par Genre:")
print(f"   - Différence: {gender_eo['difference']:.4f}")
print(f"   - Ratio: {gender_eo['ratio']:.4f}")
print(f"   - Interprétation: {gender_eo['interpretation']}")

In [ ]:
# Visualiser TPR et FPR par genre
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TPR
tpr_df = pd.DataFrame({
    'Genre': list(gender_eo['true_positive_rates'].keys()),
    'True Positive Rate': list(gender_eo['true_positive_rates'].values())
})
sns.barplot(data=tpr_df, x='Genre', y='True Positive Rate', ax=axes[0])
axes[0].set_title('True Positive Rate par Genre')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_ylim(0, 1)

# FPR
fpr_df = pd.DataFrame({
    'Genre': list(gender_eo['false_positive_rates'].keys()),
    'False Positive Rate': list(gender_eo['false_positive_rates'].values())
})
sns.barplot(data=fpr_df, x='Genre', y='False Positive Rate', ax=axes[1])
axes[1].set_title('False Positive Rate par Genre')
axes[1].set_ylabel('False Positive Rate')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 5. Audit de Fairness par Groupe d'Âge

In [ ]:
# Audit de parité démographique par groupe d'âge
age_dp = auditor.audit_demographic_parity(
    X_test,
    y_test,
    X_test['age_group']
)

print("\n📊 Audit de Parité Démographique par Groupe d'Âge:")
print(f"   - Différence: {age_dp['difference']:.4f}")
print(f"   - Ratio: {age_dp['ratio']:.4f}")
print(f"   - Interprétation: {age_dp['interpretation']}")

print("\n📊 Taux de sélection par groupe d'âge:")
for age_group, rate in age_dp['selection_rates'].items():
    print(f"   - {age_group}: {rate:.4f}")

In [ ]:
# Audit d'égalité des chances par groupe d'âge
age_eo = auditor.audit_equalized_odds(
    X_test,
    y_test,
    X_test['age_group']
)

print("\n📊 Audit d'Égalité des Chances par Groupe d'Âge:")
print(f"   - Différence: {age_eo['difference']:.4f}")
print(f"   - Ratio: {age_eo['ratio']:.4f}")
print(f"   - Interprétation: {age_eo['interpretation']}")

In [ ]:
# Visualiser TPR et FPR par groupe d'âge
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TPR
tpr_df = pd.DataFrame({
    'Groupe d\'Âge': list(age_eo['true_positive_rates'].keys()),
    'True Positive Rate': list(age_eo['true_positive_rates'].values())
})
sns.barplot(data=tpr_df, x='Groupe d\'Âge', y='True Positive Rate', ax=axes[0])
axes[0].set_title('True Positive Rate par Groupe d\'Âge')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_ylim(0, 1)

# FPR
fpr_df = pd.DataFrame({
    'Groupe d\'Âge': list(age_eo['false_positive_rates'].keys()),
    'False Positive Rate': list(age_eo['false_positive_rates'].values())
})
sns.barplot(data=fpr_df, x='Groupe d\'Âge', y='False Positive Rate', ax=axes[1])
axes[1].set_title('False Positive Rate par Groupe d\'Âge')
axes[1].set_ylabel('False Positive Rate')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 6. Audit Complet

In [ ]:
# Audit complet sur toutes les features sensibles
sensitive_features = X_test[['gender', 'age_group']].copy()
results = auditor.audit_all(X_test, y_test, sensitive_features)

print("\n✅ Audit complet terminé")

In [ ]:
# Générer le rapport d'audit
report = auditor.generate_report()
print(report)

## 7. Atténuation des Biais (Optionnel)

In [ ]:
# Appliquer une technique d'atténuation des biais
# Note: Cette étape peut prendre plusieurs minutes

mitigation = auditor.mitigate_fairness(
    X_train,
    y_train,
    X_train['gender'],
    constraint='demographic_parity',
    method='exponentiated_gradient',
    max_iter=50
)

print("\n📊 Résultats de l'atténuation:")
print(f"   - Accuracy: {mitigation['metrics']['accuracy']:.4f}")
print(f"   - DP Difference: {mitigation['metrics']['dp_difference']:.4f}")
print(f"   - EO Difference: {mitigation['metrics']['eo_difference']:.4f}")

## 8. Analyse des Métriques par Groupe

In [ ]:
# Métriques globales par groupe
overall = results['overall_metrics']

print("\n📊 Métriques globales par groupe:")

for metric_name in ['accuracy', 'precision', 'recall', 'f1']:
    print(f"\n{metric_name.upper()}:")
    metric_df = pd.DataFrame(overall.by_group[metric_name])
    print(metric_df)

In [ ]:
# Visualiser les métriques par groupe
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i, metric in enumerate(metrics):
    metric_df = pd.DataFrame(overall.by_group[metric])
    metric_df = metric_df.reset_index()
    metric_df.columns = ['Groupe', metric.capitalize()]
    
    sns.barplot(data=metric_df, x='Groupe', y=metric.capitalize(), ax=axes[i])
    axes[i].set_title(f'{metric.capitalize()} par Groupe')
    axes[i].set_ylabel(metric.capitalize())
    axes[i].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 9. Résumé

In [ ]:
print("\n" + "="*60)
print("⚖️ RÉSUMÉ DE L'AUDIT DE FAIRNESS")
print("="*60)

print("\n📊 Parité Démographique:")
print("   Genre:")
print(f"     - Différence: {gender_dp['difference']:.4f}")
print(f"     - Ratio: {gender_dp['ratio']:.4f}")
print(f"     - Interprétation: {gender_dp['interpretation']}")

print("   Groupe d'Âge:")
print(f"     - Différence: {age_dp['difference']:.4f}")
print(f"     - Ratio: {age_dp['ratio']:.4f}")
print(f"     - Interprétation: {age_dp['interpretation']}")

print("\n📊 Égalité des Chances:")
print("   Genre:")
print(f"     - Différence: {gender_eo['difference']:.4f}")
print(f"     - Ratio: {gender_eo['ratio']:.4f}")
print(f"     - Interprétation: {gender_eo['interpretation']}")

print("   Groupe d'Âge:")
print(f"     - Différence: {age_eo['difference']:.4f}")
print(f"     - Ratio: {age_eo['ratio']:.4f}")
print(f"     - Interprétation: {age_eo['interpretation']}")

print("\n📊 Recommandations:")
print("   1. Le modèle présente des disparités modérées par genre")
print("   2. Des disparités plus importantes sont observées par groupe d'âge")
print("   3. Considérer l'application de techniques d'atténuation des biais")
print("   4. Surveiller les métriques de fairness en production")

print("\n" + "="*60)
print("✅ Audit de fairness terminé avec succès")
print("="*60)